### Networking in Java
Tutto ciò che serve per la comunicazione di rete si trova nel package `java.net`. Java offre supporto per i due protocolli di trasporto principali:
- **TCP** → orientato alla connessione, affidabile (`Socket` / `ServerSocket`)
- **UDP** → senza connessione, non affidabile (`DatagramSocket` / `DatagramPacket`)

### La classe `InetAddress`
`InetAddress` rappresenta un indirizzo IP e incapsula la logica di risoluzione tra **nome host** e **indirizzo**. Non ha un costruttore pubblico: si ottengono istanze tramite metodi statici di factory.

Alcuni metodi di utilità:
- `static InetAddress getLocalHost() throws UnknownHostException` → restituisce l'indirizzo IP della macchina locale
- `static InetAddress getByName(String host) throws UnknownHostException` → restituisce l'indirizzo IP associato a un dato host name
- `static InetAddress[] getAllByName(String host) throws UnknownHostException` → restituisce un **array** di indirizzi IP per un dato hostname (un host può averne più di uno)
- `byte[] getAddress()` → restituisce l'indirizzo, contenuto nell'oggetto, come sequenza di byte
- `String getHostName()` → restituisce il nome della macchina
- `boolean isMulticastAddress()` → restituisce `true` se l'oggetto incapsula un indirizzo multicast

> *`getByName` restituisce **un** indirizzo, `getAllByName` **tutti** quelli associati al nome (es. `www.google.com` risolve a più IP per bilanciamento di carico).*

In [ ]:
import java.net.InetAddress;

// indirizzo della macchina locale
InetAddress local = InetAddress.getLocalHost();
System.out.println("getLocalHost(): " + local);
System.out.println("  getHostName(): " + local.getHostName());

// getAddress() restituisce i 4 byte (per IPv4); sono signed in Java, quindi & 0xFF
byte[] raw = local.getAddress();
StringBuilder sb = new StringBuilder("  getAddress(): ");
for (int i = 0; i < raw.length; i++) {
    sb.append(raw[i] & 0xFF);
    if (i < raw.length - 1) sb.append(".");
}
System.out.println(sb);

In [ ]:
import java.net.InetAddress;

// risoluzione di un nome → indirizzo
InetAddress one = InetAddress.getByName("localhost");
System.out.println("getByName(\"localhost\"): " + one);
System.out.println("  isMulticastAddress(): " + one.isMulticastAddress());

// un hostname può risolvere a PIÙ indirizzi
InetAddress[] all = InetAddress.getAllByName("localhost");
System.out.println("getAllByName(\"localhost\"): " + all.length + " indirizzo/i");
for (InetAddress a : all) {
    System.out.println("  - " + a.getHostAddress());
}

### TCP
La comunicazione TCP è **orientata alla connessione**: prima si stabilisce un canale tra client e server, poi i due si scambiano un flusso (*stream*) di byte. I ruoli sono asimmetrici:
- il **client** apre attivamente la connessione verso un host/porta noti, usando `Socket`
- il **server** resta in ascolto su una porta e accetta le connessioni in arrivo, usando `ServerSocket`

#### Lato client: `Socket`
Il costruttore più comune è:

```java
public Socket(String host, int port) throws UnknownHostException, IOException
```

Crea una stream socket e la **connette** alla porta indicata sull'host specificato (per nome o per IP testuale, es. `"localhost"` o `"127.0.0.1"`).

> *Se `host` è `null`, equivale a `InetAddress.getByName(null)`, cioè all'indirizzo di **loopback**.*

> **Nota sugli overload**: `Socket` ha più costruttori. Oltre a quello con `String`, esiste `Socket(InetAddress address, int port)` che prende un indirizzo **già risolto**. La versione con `String` è solo più comoda: internamente risolve l'host con `InetAddress.getByName(host)`. Le due forme sono equivalenti.

#### Lato server: `ServerSocket`
Il server usa `ServerSocket(int port)` oppure `ServerSocket(int port, int backlog)`:

```java
public ServerSocket(int port, int backlog) throws IOException
```

Crea una server socket e la **lega** (*bind*) alla porta locale indicata.

- `port` → numero di porta; con **`0`** il sistema ne assegna una automaticamente (porta *ephemeral*), recuperabile poi con `getLocalPort()`.
- `backlog` → lunghezza massima della **coda** delle richieste di connessione in attesa di essere accettate. Se arriva una connessione a coda piena, viene **rifiutata**.

> *Il `backlog` è un "suggerimento": la semantica esatta dipende dall'implementazione, che può imporre un massimo o ignorarlo. Se il valore è ≤ 0 si usa un default specifico dell'implementazione.*

Il metodo chiave è `accept()`: è **bloccante** e resta in attesa finché un client non si connette, restituendo poi una nuova `Socket` dedicata a quel client.

#### `getPort()` vs `getLocalPort()`
Ogni socket connessa ha **due estremità**, ciascuna con la sua coppia (IP, porta): quella locale e quella del peer remoto.

- `getPort()` → la porta **remota** (del peer dall'altro lato)
- `getLocalPort()` → la porta **locale** (di questa estremità)

| Classe | `getPort()` | `getLocalPort()` |
|---|---|---|
| `Socket` (lato client) | porta del server | porta *ephemeral* assegnata dal SO al client |
| `Socket` (da `accept()`, lato server) | porta *ephemeral* del client | porta di ascolto del server |
| `ServerSocket` | — (non esiste) | porta su cui è in ascolto |
| `DatagramSocket` (UDP) | porta del peer **solo se** è stata fatta `connect()`, altrimenti `-1` | porta locale della socket |

/

> *Per questo, su una `DatagramSocket` non connessa (il caso tipico del server UDP) si usa `getLocalPort()`: `getPort()` restituirebbe `-1`.*

In [ ]:
import java.net.ServerSocket;
import java.net.Socket;

// porta 0 → il SO sceglie una porta libera
ServerSocket server = new ServerSocket(0);
int serverPort = server.getLocalPort();
System.out.println("ServerSocket in ascolto su porta: " + serverPort);

// il client si connette a quella porta
Socket client = new Socket("localhost", serverPort);
Socket accepted = server.accept();   // socket lato server per QUESTO client

System.out.println("--- lato client ---");
System.out.println("  getPort()      = " + client.getPort()       + "  (porta del server)");
System.out.println("  getLocalPort() = " + client.getLocalPort()  + "  (porta ephemeral del client)");

System.out.println("--- lato server (socket da accept) ---");
System.out.println("  getPort()      = " + accepted.getPort()      + "  (porta ephemeral del client)");
System.out.println("  getLocalPort() = " + accepted.getLocalPort() + "  (porta di ascolto del server)");

client.close();
accepted.close();
server.close();

#### Workflow completo
L'idea è: il client si connette, entrambi avvolgono gli stream della socket in stream di più alto livello (qui `DataInputStream`/`DataOutputStream` per leggere/scrivere stringhe via `writeUTF`/`readUTF`), si scambiano i dati, e infine chiudono tutto.

Il server adotta il pattern **thread-per-connection**: fa solo `accept()` in loop e delega ogni connessione a un `Thread` dedicato, restando libero di accettare nuovi client.

> Nell'esempio seguente client e server girano nello **stesso processo** (su due thread) così da poterlo eseguire in un'unica cella; in un caso reale sarebbero due programmi separati.

In [ ]:
import java.net.ServerSocket;
import java.net.Socket;
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.IOException;

int PORT = 2222;

// --- worker che serve un singolo client ---
public class MyWorker extends Thread {
    private Socket s;

    MyWorker(Socket socket) {
        s = socket;
    }

    public void run() {
        try {
            DataInputStream fromClient = new DataInputStream(s.getInputStream());
            DataOutputStream toClient = new DataOutputStream(s.getOutputStream());

            String message = fromClient.readUTF();
            System.out.println("[Server] ricevuto: " + message);
            toClient.writeUTF("Hello client!");

            // è importante chiudere gli stream e la connessione
            fromClient.close();
            toClient.close();
            s.close();
        } catch (IOException e) {
            e.printStackTrace();
        }
    }
}

// --- server: accept loop su un thread separato (qui accetta 1 client poi esce) ---
public class ServerLoop extends Thread {
    public void run() {
        try {
            ServerSocket server = new ServerSocket(PORT);
            Socket client = server.accept();   // bloccante: aspetta un client
            new MyWorker(client).start();      // un thread per ogni client
            server.close();
        } catch (IOException e) {
            e.printStackTrace();
        }
    }
}

ServerLoop server = new ServerLoop();
server.start();
Thread.sleep(200);   // do tempo al server di mettersi in ascolto

// --- client ---
Socket conn = new Socket("localhost", PORT);
DataOutputStream toServer = new DataOutputStream(conn.getOutputStream());
DataInputStream fromServer = new DataInputStream(conn.getInputStream());

toServer.writeUTF("Hello server!");
String response = fromServer.readUTF();
System.out.println("[Client] risposta: " + response);

toServer.close();
fromServer.close();
conn.close();
server.join();

### UDP
La comunicazione UDP è **senza connessione**: non c'è handshake né canale stabile, si inviano singoli **datagrammi** indipendenti, senza garanzia di consegna né di ordine. Le classi coinvolte sono due:
- **`DatagramSocket`** → la socket da cui si inviano e ricevono i datagrammi
- **`DatagramPacket`** → il singolo pacchetto (dati + eventuale destinazione)

#### `DatagramSocket`
```java
public DatagramSocket() throws SocketException
```

Costruisce una datagram socket e la lega a una **qualsiasi porta disponibile** sulla macchina locale (sul *wildcard address*, un IP scelto dal kernel). Per legarla a una porta specifica si usa `DatagramSocket(int port)`.

#### `DatagramPacket`
A differenza di TCP, qui la destinazione **non** è nella socket ma nel singolo pacchetto. Ci sono quindi due costruttori a seconda dello scopo.

**Per inviare** (si specifica anche destinazione e porta):
```java
public DatagramPacket(byte[] buf, int length, InetAddress address, int port)
```

| Parametro | Significato |
|---|---|
| `buf` | i dati del pacchetto |
| `length` | la lunghezza dei dati (deve essere ≤ `buf.length`) |
| `address` | l'indirizzo di destinazione |
| `port` | la porta di destinazione |

**Per ricevere** (solo il buffer in cui scrivere i dati in arrivo):
```java
public DatagramPacket(byte[] buf, int length)
```

#### Workflow completo
> *Si osservi come il client **non** specifichi indirizzo e porta di destinazione nella socket, ma direttamente nel pacchetto da mandare: è proprio il principio di UDP, cioè usare la **stessa socket** per mandare pacchetti a destinatari diversi.*

Anche qui, per poterlo eseguire in un'unica cella, server e client girano sullo stesso processo su due thread.

In [ ]:
import java.net.DatagramSocket;
import java.net.DatagramPacket;
import java.net.InetAddress;
import java.io.IOException;

int PORT = 3333;

// --- server: si mette in ascolto e riceve un pacchetto ---
public class UdpServer extends Thread {
    public void run() {
        try {
            DatagramSocket socket = new DatagramSocket(PORT);
            System.out.println("[Server] in ascolto su porta: " + socket.getLocalPort());

            byte[] data = new byte[65508];   // dimensione massima di un datagramma UDP
            DatagramPacket pkt = new DatagramPacket(data, data.length);
            socket.receive(pkt);             // bloccante: aspetta un pacchetto

            // si legge solo la porzione valida: da offset 0 per getLength() byte
            String received = new String(pkt.getData(), 0, pkt.getLength());
            System.out.println("[Server] ricevuto: " + received);
            socket.close();
        } catch (IOException e) {
            e.printStackTrace();
        }
    }
}

UdpServer server = new UdpServer();
server.start();
Thread.sleep(200);

// --- client: invia un pacchetto, senza connessione ---
DatagramSocket socket = new DatagramSocket();
InetAddress addr = InetAddress.getByName("localhost");

String s = new String("Hello Server!");
DatagramPacket pkt = new DatagramPacket(s.getBytes(), s.getBytes().length, addr, PORT);
socket.send(pkt);

socket.close();
server.join();